# Analyse Exploratoire — PaySim (Mobile Money Simulation)

Ce notebook réalise l'analyse exploratoire du jeu de données **PaySim**, un simulateur de
transactions de paiement mobile générant ~6,3 millions de transactions dont une faible
proportion sont frauduleuses.

**Colonnes disponibles :** `step`, `type`, `amount`, `nameOrig`, `oldbalanceOrg`,
`newbalanceOrig`, `nameDest`, `oldbalanceDest`, `newbalanceDest`, `isFraud`, `isFlaggedFraud`

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ajouter le répertoire racine au path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.loader import load_paysim, get_dataset_info
from config import FIGURES_DIR, FIGURE_DPI, FIGURE_SIZE_SINGLE

# Style graphique
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = FIGURE_DPI

# Répertoire de sauvegarde
EDA_DIR = os.path.join(FIGURES_DIR, 'eda')
os.makedirs(EDA_DIR, exist_ok=True)

# Chargement du jeu de données
try:
    df = load_paysim()
    print(f'Dataset chargé avec succès : {df.shape[0]:,} lignes, {df.shape[1]} colonnes')
except FileNotFoundError as e:
    print(f'ERREUR : {e}')
    print('Veuillez télécharger le fichier depuis :')
    print('https://www.kaggle.com/datasets/ealaxi/paysim1')
    raise

In [ ]:
# Aperçu global
print('='*60)
print(f'Dimensions : {df.shape}')
print('='*60)
print()

# Premières lignes
display(df.head(10))

print()
print('--- Types et valeurs non nulles ---')
print(df.info())

print()
print('--- Statistiques descriptives ---')
display(df.describe().round(2))

# Résumé via notre utilitaire
info = get_dataset_info(df)
print(f"\nTaux de fraude : {info['fraud_rate']*100:.4f}%")
print(f"Ratio déséquilibre (légit/fraude) : {info['imbalance_ratio']:.0f}:1")
print(f"Mémoire utilisée : {info['memory_mb']:.1f} Mo")

## Distribution des Classes

Le jeu de données PaySim est fortement déséquilibré : les transactions frauduleuses
représentent une fraction minuscule de l'ensemble des transactions.

In [ ]:
# Distribution de la variable cible isFraud
class_counts = df['isFraud'].value_counts()
class_pct = df['isFraud'].value_counts(normalize=True) * 100

print('Distribution des classes :')
for label, count in class_counts.items():
    pct = class_pct[label]
    name = 'Fraude' if label == 1 else 'Légitime'
    print(f'  {name} ({label}): {count:>10,}  ({pct:.4f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barplot
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(['Légitime (0)', 'Fraude (1)'], class_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Distribution des Classes — PaySim', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Nombre de transactions')
for bar, count in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{count:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Légitime', 'Fraude'],
            autopct='%1.3f%%', colors=colors, startangle=90,
            explode=[0, 0.1], shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Proportion des Classes', fontsize=14, fontweight='bold')

plt.tight_layout()
fig.savefig(os.path.join(EDA_DIR, 'paysim_class_distribution.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f'Figure sauvegardée : {os.path.join(EDA_DIR, "paysim_class_distribution.png")}')

## Distribution par Type de Transaction

PaySim contient 5 types de transactions : CASH_IN, CASH_OUT, DEBIT, PAYMENT, TRANSFER.
Les fraudes se concentrent exclusivement sur les types **CASH_OUT** et **TRANSFER**.

In [ ]:
# Taux de fraude par type de transaction
fraud_by_type = df.groupby('type').agg(
    total=('isFraud', 'count'),
    n_fraud=('isFraud', 'sum')
).reset_index()
fraud_by_type['fraud_rate'] = (fraud_by_type['n_fraud'] / fraud_by_type['total'] * 100)

print('Fraude par type de transaction :')
display(fraud_by_type.sort_values('fraud_rate', ascending=False))

# Grouped bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Nombre de transactions par type (log)
type_counts = df['type'].value_counts().sort_index()
fraud_type_counts = df[df['isFraud']==1]['type'].value_counts().sort_index()

x = np.arange(len(type_counts))
width = 0.35

bars1 = axes[0].bar(x - width/2, type_counts.values, width, label='Total', color='#3498db', edgecolor='black')
fraud_vals = [fraud_type_counts.get(t, 0) for t in type_counts.index]
bars2 = axes[0].bar(x + width/2, fraud_vals, width, label='Fraude', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Type de Transaction')
axes[0].set_ylabel('Nombre (échelle log)')
axes[0].set_title('Transactions Totales vs Frauduleuses par Type', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(type_counts.index, rotation=45)
axes[0].set_yscale('log')
axes[0].legend()

# Taux de fraude par type
bars3 = axes[1].bar(fraud_by_type['type'], fraud_by_type['fraud_rate'],
                     color=['#e74c3c' if r > 0 else '#2ecc71' for r in fraud_by_type['fraud_rate']],
                     edgecolor='black')
axes[1].set_xlabel('Type de Transaction')
axes[1].set_ylabel('Taux de Fraude (%)')
axes[1].set_title('Taux de Fraude par Type de Transaction', fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
for bar, rate in zip(bars3, fraud_by_type['fraud_rate']):
    if rate > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                     f'{rate:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
fig.savefig(os.path.join(EDA_DIR, 'paysim_fraud_by_type.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Les fraudes ne se produisent que sur CASH_OUT et TRANSFER.')

## Analyse des Montants

Examinons la distribution des montants de transaction pour les transactions légitimes
et frauduleuses.

In [ ]:
# Distribution des montants par statut de fraude
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogramme en échelle log
df_legit = df[df['isFraud'] == 0]['amount']
df_fraud = df[df['isFraud'] == 1]['amount']

axes[0].hist(df_legit, bins=100, alpha=0.7, label='Légitime', color='#2ecc71', log=True)
axes[0].hist(df_fraud, bins=100, alpha=0.7, label='Fraude', color='#e74c3c', log=True)
axes[0].set_xlabel('Montant')
axes[0].set_ylabel('Fréquence (échelle log)')
axes[0].set_title('Distribution des Montants (échelle log)', fontsize=13, fontweight='bold')
axes[0].legend()

# Boxplot comparatif
fraud_labels = df['isFraud'].map({0: 'Légitime', 1: 'Fraude'})
box_data = pd.DataFrame({'Montant': df['amount'], 'Classe': fraud_labels})
sns.boxplot(data=box_data, x='Classe', y='Montant', ax=axes[1],
            palette={'Légitime': '#2ecc71', 'Fraude': '#e74c3c'})
axes[1].set_yscale('log')
axes[1].set_title('Boxplot des Montants par Classe (échelle log)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Montant (échelle log)')

plt.tight_layout()
fig.savefig(os.path.join(EDA_DIR, 'paysim_amount_distribution.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

# Statistiques
print('Statistiques des montants :')
print(f'  Légitimes — Moyenne: {df_legit.mean():,.2f}, Médiane: {df_legit.median():,.2f}, Max: {df_legit.max():,.2f}')
print(f'  Fraudes   — Moyenne: {df_fraud.mean():,.2f}, Médiane: {df_fraud.median():,.2f}, Max: {df_fraud.max():,.2f}')

## Analyse des Soldes

Les variations de solde (avant vs après transaction) révèlent des patterns
caractéristiques des transactions frauduleuses.

In [ ]:
# Patterns de changement de solde
df['balance_change_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Échantillonnage pour la visualisation (le dataset complet est trop grand)
sample_legit = df[df['isFraud'] == 0].sample(n=min(5000, len(df[df['isFraud']==0])), random_state=42)
sample_fraud = df[df['isFraud'] == 1].sample(n=min(5000, len(df[df['isFraud']==1])), random_state=42)

# Scatter : montant vs changement de solde (origine)
axes[0].scatter(sample_legit['amount'], sample_legit['balance_change_orig'],
                alpha=0.3, s=5, label='Légitime', color='#2ecc71')
axes[0].scatter(sample_fraud['amount'], sample_fraud['balance_change_orig'],
                alpha=0.5, s=10, label='Fraude', color='#e74c3c')
axes[0].set_xlabel('Montant')
axes[0].set_ylabel('Changement de Solde (Origine)')
axes[0].set_title('Montant vs Changement de Solde — Origine', fontsize=13, fontweight='bold')
axes[0].legend()

# Distribution du changement de solde
axes[1].hist(sample_legit['balance_change_orig'], bins=80, alpha=0.6,
             label='Légitime', color='#2ecc71', log=True)
axes[1].hist(sample_fraud['balance_change_orig'], bins=80, alpha=0.6,
             label='Fraude', color='#e74c3c', log=True)
axes[1].set_xlabel('Changement de Solde (Origine)')
axes[1].set_ylabel('Fréquence (log)')
axes[1].set_title('Distribution du Changement de Solde', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(EDA_DIR, 'paysim_balance_patterns.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

# Observation clé
fraud_zero_balance = (df[df['isFraud']==1]['newbalanceOrig'] == 0).mean() * 100
print(f'\nObservation : {fraud_zero_balance:.1f}% des fraudes vident entièrement le solde d\'origine (newbalanceOrig = 0)')

## Feature Engineering Preview

Création de variables dérivées qui capturent les patterns identifiés dans l'analyse :
- **Amount_Ratio** : ratio montant / solde initial (indicateur de fraude si > 1)
- **Balance_Change_Orig** : variation du solde de l'expéditeur
- **Balance_Change_Dest** : variation du solde du destinataire

In [ ]:
# Prévisualisation du feature engineering
from src.data.preprocessor import FraudPreprocessor

preprocessor = FraudPreprocessor()
df_eng = preprocessor.engineer_features_paysim(df)

# Afficher les nouvelles colonnes
new_cols = ['Amount_Ratio', 'Balance_Change_Orig', 'Balance_Change_Dest', 'Type_Encoded', 'Amount_Log']
print('Nouvelles colonnes créées :')
display(df_eng[new_cols].describe().round(2))

print()
print('Aperçu avec les nouvelles features :')
display(df_eng[['type', 'amount', 'isFraud'] + new_cols].head(10))

# Corrélation entre Amount_Ratio et fraude
fig, ax = plt.subplots(figsize=(10, 5))
sample_eng = df_eng.sample(n=min(10000, len(df_eng)), random_state=42)
colors_map = sample_eng['isFraud'].map({0: '#2ecc71', 1: '#e74c3c'})
ax.scatter(sample_eng['Amount_Ratio'].clip(upper=10), sample_eng['Balance_Change_Orig'],
           c=colors_map, alpha=0.4, s=5)
ax.set_xlabel('Amount_Ratio (plafonné à 10)')
ax.set_ylabel('Balance_Change_Orig')
ax.set_title('Amount_Ratio vs Balance_Change_Orig (vert=légitime, rouge=fraude)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(EDA_DIR, 'paysim_feature_engineering_preview.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

## Résumé

### Observations clés de l'analyse exploratoire PaySim :

1. **Déséquilibre extrême** : Le taux de fraude est d'environ 0,13%, soit un ratio d'environ 773:1 entre transactions légitimes et frauduleuses.

2. **Types ciblés** : 100% des fraudes se produisent sur seulement 2 types de transactions : **CASH_OUT** et **TRANSFER**. Les types CASH_IN, DEBIT et PAYMENT ne contiennent aucune fraude.

3. **Montants** : Les transactions frauduleuses ont tendance à impliquer des montants plus élevés que les transactions légitimes, en particulier pour les transferts.

4. **Pattern de vidage de compte** : La grande majorité des fraudes vident entièrement le solde de l'expéditeur (`newbalanceOrig = 0`), ce qui constitue un signal fort.

5. **Variables dérivées prometteuses** :
   - `Amount_Ratio` (montant / solde initial) — les valeurs proches de 1 ou supérieures sont suspectes
   - `Balance_Change_Orig` — les changements brusques et totaux sont caractéristiques de la fraude
   - `Balance_Change_Dest` — les destinataires de fraude montrent des patterns d'accumulation

6. **`isFlaggedFraud`** : Ce flag natif du simulateur est très conservateur et ne capture qu'une infime fraction des fraudes réelles, confirmant la nécessité d'un modèle ML.

### Implications pour la modélisation :
- Les techniques de rééquilibrage (SMOTE, undersampling) seront indispensables
- Les features de type, de ratio de montant et de changement de solde seront les plus discriminantes
- Le filtrage par type (CASH_OUT, TRANSFER uniquement) pourrait simplifier le problème